# Stage 4a: Cross-Play Data Generation

**Goal:** Generate high-quality synthetic conversations using **cross-play** —
two independent Moshi teacher models converse in full-duplex mode. This replaces
single-model self-play which produces low-quality, repetitive audio.

**What's captured per step (for offline distillation):**
- **Top-1000 text logits** (values + indices) → KL-divergence soft target matching
- **Top-1000 audio CB0 logits** (values + indices) → Audio KD loss
- **Hard text tokens** (Inner Monologue) → Hard label cross-entropy
- **Hard audio tokens** (all 8 codebooks) → Codebook KD loss
- **User audio tokens** (Channel A) → Input context for training

**Output:** 2 `.npz` files per conversation (one per model), same format as
self-play data → compatible with existing `SelfPlayDataset`.

**Run on:** Colab Pro (**A100 required** — two teacher models ≈ 31 GB VRAM)

**Initial target:** 4 hours of conversation (~48 convos × 5 min). Expandable —
run more sessions to incrementally append data.

## Cell 1: Session Startup

In [ ]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)

# Check GPU — A100 required for cross-play (two teacher models)
import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")

if vram_gb < 35:
    print(f"\n⚠️  WARNING: Cross-play needs ~31 GB VRAM for two teacher models.")
    print(f"   You have {vram_gb:.0f} GB. Switch to A100 runtime.")
    print(f"   Go to: Runtime → Change runtime type → A100")

## Cell 2: Load Two Moshi Teachers

Loads the teacher weights **twice** into two independent `LMModel` + `LMGen`
instances. Both use identical sampling parameters.

In [ ]:
import torch
import json
from pathlib import Path
from moshi.models import loaders
from moshi.models.lm import LMGen

WEIGHTS_DIR = "/content/drive/MyDrive/moshilite/upstream_weights/moshiko"

# Detect GPU capability for dtype
gpu_name = torch.cuda.get_device_name(0).lower()
if any(x in gpu_name for x in ["a100", "h100", "l4", "l40"]):
    DTYPE = torch.bfloat16
    print(f"Using bfloat16 (detected {gpu_name})")
else:
    DTYPE = torch.float16
    print(f"Using float16 (detected {gpu_name})")

weights_path = Path(WEIGHTS_DIR)
if not weights_path.exists():
    raise FileNotFoundError(f"Cannot find weights dir: {weights_path}")

config_path = weights_path / "config.json"
moshi_name = "model.safetensors"
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    moshi_name = config.get("moshi_name", moshi_name)
else:
    config = None

model_path = weights_path / moshi_name
if not model_path.exists():
    sf_files = list(weights_path.glob("*.safetensors"))
    sf_files = [f for f in sf_files if "tokenizer" not in f.name.lower()
                and "mimi" not in f.name.lower()]
    if not sf_files:
        raise FileNotFoundError(f"No LM weights in {weights_path}")
    model_path = sf_files[0]

# ── Load Model A ──
print("\nLoading Model A (teacher)...")
lm_model_a = loaders.get_moshi_lm(
    model_path, lm_kwargs=config, device="cuda", dtype=DTYPE
)
lm_model_a.eval()
params_a = sum(p.numel() for p in lm_model_a.parameters()) / 1e9
print(f"  Model A: {params_a:.2f}B params")
print(f"  VRAM after Model A: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Load Model B (second independent copy) ──
print("\nLoading Model B (teacher, independent copy)...")
lm_model_b = loaders.get_moshi_lm(
    model_path, lm_kwargs=config, device="cuda", dtype=DTYPE
)
lm_model_b.eval()
params_b = sum(p.numel() for p in lm_model_b.parameters()) / 1e9
print(f"  Model B: {params_b:.2f}B params")
print(f"  VRAM after both models: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Create LMGen wrappers ──
TEMPERATURE = 0.8
TEMPERATURE_TEXT = 0.7

lm_gen_a = LMGen(
    lm_model_a, use_sampling=True,
    temp=TEMPERATURE, temp_text=TEMPERATURE_TEXT,
    top_k=250, top_k_text=25,
)
lm_gen_b = LMGen(
    lm_model_b, use_sampling=True,
    temp=TEMPERATURE, temp_text=TEMPERATURE_TEXT,
    top_k=250, top_k_text=25,
)

print(f"\n✅ Both models loaded. Ready for cross-play.")
print(f"   n_q={lm_model_a.n_q}, dep_q={lm_model_a.dep_q}, dim={lm_model_a.dim}")
print(f"   card={lm_model_a.card}, text_card={lm_model_a.text_card}")

## Cell 3: Configuration & Sanity Test

Configure generation parameters, then run one short cross-play
conversation to verify everything works and measure speed.

In [ ]:
# ============================================================
# ⚙️  CONFIGURATION — CHANGE THESE PER PERSON / SESSION
# ============================================================
PERSON_ID = "A"              # Unique per person: A, B, C, D...
SESSION_NUM = 0              # Increment each completed session: 0, 1, 2...
CONVOS_PER_SESSION = 48      # ~48 for 4 hours of audio (5 min each)

# ── Generation parameters ──
STEPS_PER_CONV = 3750        # 5 minutes @ 12.5 Hz
TOP_K_LOGITS = 1000          # Top-k logits to save for offline distillation
SEED_TYPE = "noise"          # Default seed type (80% noise, 10% acoustic, 10% silence)
REPETITION_PENALTY = 1.3     # Text repetition penalty (>1.0 penalizes repeats)
REP_WINDOW = 50              # Sliding window for repetition tracking

# ── Auto-computed — DO NOT EDIT ──
BATCH_ID = f"xplay_{PERSON_ID}_{SESSION_NUM:02d}"   # e.g. xplay_A_00
OUTPUT_DIR = "/content/drive/MyDrive/moshilite/self_play_data/conversations"

print(f"👤 Person: {PERSON_ID}")
print(f"📦 Batch:  {BATCH_ID}")
print(f"🎯 Target: {CONVOS_PER_SESSION} conversations × 5 min = "
      f"{CONVOS_PER_SESSION * 5 / 60:.1f} hours of audio")
print(f"   → {CONVOS_PER_SESSION * 2} training examples "
      f"({CONVOS_PER_SESSION * 10 / 60:.1f} hrs training data)")
print(f"📂 Output: {OUTPUT_DIR}/{BATCH_ID}/")
print(f"🔑 Top-k logits: {TOP_K_LOGITS}")
print()

# ── Quick sanity test (short cross-play conversation) ──
import time
from moshilite.self_play.generator import generate_cross_play_conversation

print("Generating test cross-play conversation (50 steps ≈ 4 seconds)...")
t0 = time.time()
with torch.no_grad():
    test_record = generate_cross_play_conversation(
        lm_gen_a=lm_gen_a,
        lm_gen_b=lm_gen_b,
        num_steps=50,
        seed_type="noise",
        seed_index=0,
        top_k_logits=TOP_K_LOGITS,
        conv_id="test_000",
        card=lm_model_a.card,
        device="cuda",
        repetition_penalty=REPETITION_PENALTY,
        rep_window=REP_WINDOW,
    )
elapsed = time.time() - t0
ms_per_step = (elapsed / 50) * 1000

print(f"\n✅ Test passed: {test_record.num_valid_steps} valid steps in {elapsed:.1f}s")
print(f"   Speed: {ms_per_step:.0f} ms/step")
print(f"   Model A: text_tokens={test_record.a_text_tokens.shape}, "
      f"audio_tokens={test_record.a_audio_tokens.shape}, "
      f"logits={test_record.a_text_logits_vals.shape}")
print(f"   Model B: text_tokens={test_record.b_text_tokens.shape}, "
      f"audio_tokens={test_record.b_audio_tokens.shape}, "
      f"logits={test_record.b_text_logits_vals.shape}")

# Estimate session time
est_hours = (STEPS_PER_CONV * CONVOS_PER_SESSION * 1.4 * ms_per_step / 1000) / 3600
print(f"\n⏱️  Estimated session time: {est_hours:.1f} hours "
      f"(with ~29% rejection overhead)")

## Cell 4: Generate Full Batch

Runs `generate_cross_play_batch()` which:
1. Two models converse at each step (Model A hears B, Model B hears A)
2. Captures top-1000 logits + hard tokens from **both** models
3. Quality-filters both streams (rejects if either fails)
4. Saves 2 `.npz` files per conversation (backward-compatible with SelfPlayDataset)
5. Supports resume — re-run with same BATCH_ID to continue after crash

**Saved per `.npz` (for offline distillation):**
- `text_logits_vals/idxs` [T, 1000] — sparse top-1000 text logits for KL-div KD
- `audio_cb0_logits_vals/idxs` [T, 1000] — sparse top-1000 audio CB0 logits for KL-div KD
- `text_tokens` [T] — hard text labels for CE loss
- `audio_tokens` [8, T] — hard audio labels (all 8 codebooks) for codebook KD
- `user_audio_tokens` [8, T] — input context (other model's audio)

In [ ]:
from moshilite.self_play.generator import generate_cross_play_batch

print(f"🚀 Starting cross-play generation: {CONVOS_PER_SESSION} conversations, "
      f"{STEPS_PER_CONV} steps each (5 min)")
print(f"📦 Batch: {BATCH_ID}")
print(f"📂 Output: {OUTPUT_DIR}/{BATCH_ID}/")
print(f"🔑 Top-k logits: {TOP_K_LOGITS}")
print(f"💾 Each .npz saves: top-{TOP_K_LOGITS} text logits, top-{TOP_K_LOGITS} audio CB0 logits,")
print(f"   hard text tokens, hard audio tokens (8 CBs), user audio tokens")
print()

stats = generate_cross_play_batch(
    lm_gen_a=lm_gen_a,
    lm_gen_b=lm_gen_b,
    num_conversations=CONVOS_PER_SESSION,
    steps_per_conversation=STEPS_PER_CONV,
    top_k_logits=TOP_K_LOGITS,
    output_dir=OUTPUT_DIR,
    batch_id=BATCH_ID,
    start_index=0,
    card=lm_model_a.card,
    device="cuda",
    repetition_penalty=REPETITION_PENALTY,
    rep_window=REP_WINDOW,
)

## Cell 5: Verify & Report

In [ ]:
import json
from pathlib import Path
import numpy as np

batch_dir = Path(OUTPUT_DIR) / BATCH_ID

# Load batch metadata
with open(batch_dir / "batch_meta.json") as f:
    meta = json.load(f)

print(f"📊 Batch {BATCH_ID} Summary:")
print(f"   Mode: {meta.get('mode', 'cross_play')}")
print(f"   Accepted: {meta['accepted']}")
print(f"   Rejected: {meta['rejected']}")
print(f"   Acceptance rate: {meta['acceptance_rate']:.1%}")
print(f"   Audio duration: {meta['total_audio_hours']:.2f} hours")
print(f"   Training data: {meta.get('total_training_hours', meta['total_audio_hours'] * 2):.2f} hours")
print(f"   Wall time: {meta['total_wall_time_s'] / 60:.1f} min")
print(f"   Top-k logits: {meta.get('top_k_logits', 'N/A')}")
print(f"   Rejection reasons: {meta.get('rejection_reasons', {})}")

# Count .npz files (both modelA and modelB)
npz_a = sorted(batch_dir.glob("*_modelA.npz"))
npz_b = sorted(batch_dir.glob("*_modelB.npz"))
print(f"\n📂 {len(npz_a)} modelA + {len(npz_b)} modelB = {len(npz_a) + len(npz_b)} .npz files")

# Inspect a sample file
if npz_a:
    sample = np.load(str(npz_a[0]))
    print(f"\nSample file contents ({npz_a[0].name}):")
    for k in sample.files:
        arr = sample[k]
        print(f"   {k}: shape={arr.shape}, dtype={arr.dtype}")

    total_size_mb = sum(
        f.stat().st_size for f in batch_dir.glob("*.npz")
    ) / 1e6
    print(f"\n💾 Total disk usage: {total_size_mb:.1f} MB")

## Cell 6: Merge & Verify (Lead Person)

Run this after collecting all batch folders from team members.
The `SelfPlayDataset` auto-discovers all `.npz` files via `rglob`.

In [ ]:
# ============================================================
# 🔗 MERGE & VERIFY (Run by lead person after collecting batches)
# ============================================================
from pathlib import Path
import json

data_dir = Path("/content/drive/MyDrive/moshilite/self_play_data/conversations")
all_npz = sorted(data_dir.rglob("*.npz"))
all_npz = [f for f in all_npz if "rejected" not in f.parts]

# Categorize: self-play vs cross-play
selfplay_files = [f for f in all_npz if "_model" not in f.stem]
xplay_a_files = [f for f in all_npz if f.stem.endswith("_modelA")]
xplay_b_files = [f for f in all_npz if f.stem.endswith("_modelB")]

# Group by batch
batches = {}
for f in all_npz:
    batch = f.parent.name
    batches.setdefault(batch, []).append(f)

print(f"📊 DATASET SUMMARY")
print(f"{'='*70}")
total_hours = 0
for batch, files in sorted(batches.items()):
    # Each file = 5 min of training data
    hours = len(files) * 5 / 60
    total_hours += hours
    meta_path = data_dir / batch / "batch_meta.json"
    mode = "?"
    acc_rate = "n/a"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        mode = meta.get('mode', 'self_play')
        acc_rate = f"{meta.get('acceptance_rate', 0):.0%}"
    print(f"  {batch:25s}: {len(files):4d} files × 5 min = {hours:5.1f} hrs  "
          f"(mode: {mode}, acceptance: {acc_rate})")

print(f"{'='*70}")
print(f"  Self-play files:  {len(selfplay_files)}")
print(f"  Cross-play files: {len(xplay_a_files)} modelA + {len(xplay_b_files)} modelB")
print(f"  TOTAL: {len(all_npz)} training examples = {total_hours:.1f} hours")
print(f"\n{'✅ Ready for training!' if total_hours >= 4 else f'⏳ Need more data to reach target'}")